# 03 — Results

Every result of the study, read from saved files only. This notebook never runs the experiment
and never opens the test set: `python -m fraudstudy.experiments` wrote the cell files,
`python -m fraudstudy.evaluation --test` wrote `test_results.json` (the one opening of the test
set) and `--stats` wrote `stats.json`. Each section below says what it shows and how to read it.
Figures are saved to `artifacts/figures/`.

**Setup.** Imports, the data source and the saved results. `SOURCE = "sample"` reads the trial run
on the fake data in `artifacts/sample/` instead; its numbers mean nothing and are never committed.
`models` lists the models that have finished cells, so the notebook also works while some are
still running.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from fraudstudy.evaluation import load_cells
from fraudstudy.experiments import MODELS, TECHNIQUES, artifacts_dir

SOURCE = "raw"  # "raw" = the real results, "sample" = the fake copy in artifacts/sample/
METRICS_DIR = artifacts_dir(SOURCE) / "metrics"
FIGURES_DIR = artifacts_dir(SOURCE) / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
LABELS = {"baseline": "A baseline", "ros": "B ROS", "rus": "C RUS", "smote": "D SMOTE",
          "smote_rus": "E SMOTE+RUS", "class_weight": "F class weight"}


def read_json(name: str) -> dict | None:
    """One saved results file from METRICS_DIR, or None if it does not exist yet."""
    path = METRICS_DIR / name
    return json.loads(path.read_text()) if path.is_file() else None


cells = load_cells(SOURCE)  # every finished cell, from experiments.py
test_results = read_json("test_results.json")  # from evaluation.py --test
stats = read_json("stats.json")  # from evaluation.py --stats
eda_summary = read_json("eda_summary.json")  # from notebooks/01_eda.ipynb
models = [m for m in MODELS if any(cell["model"] == m for cell in cells.values())]
print(f"{len(cells)} cells loaded from {METRICS_DIR}; models: {', '.join(models)}")
if test_results is None:
    print("test_results.json is missing: run python -m fraudstudy.evaluation --test first.")

## 1. The data in numbers
How many payments, how many frauds, and how the 70/30 split divides them. The numbers come from
the EDA summary; when it is missing (as on the fake sample), they are rebuilt from one cell:
its training class counts plus the four counts of its test confusion matrix. The rebuilt
numbers are after duplicate removal, so they can be a little smaller than the EDA ones.

In [ ]:
if eda_summary is not None:
    rows, frauds = eda_summary["rows"], eda_summary["frauds"]
    train_rows, test_rows = eda_summary["train_rows"], eda_summary["test_rows"]
else:
    name = next(iter(cells))  # any cell: they all share the same split
    before, test = cells[name]["train_counts_before"], test_results[name]["test"]
    train_rows = before["legit"] + before["fraud"]
    test_rows = test["tp"] + test["fp"] + test["tn"] + test["fn"]
    rows = train_rows + test_rows
    frauds = before["fraud"] + test["tp"] + test["fn"]  # frauds = caught + missed in the test
fraud_rate = frauds / rows
print(f"payments:   {rows:,}")
print(f"frauds:     {frauds:,}")
print(f"fraud rate: {fraud_rate:.4%} (1 fraud for every {round((rows - frauds) / frauds)} legit)")
print(f"training:   {train_rows:,} rows    test: {test_rows:,} rows")

## 2. What each technique does to the training data
One panel per model: the bars are the training class counts the model was fitted on after the
technique (at the ratio the search picked), the dashed lines the counts before. The y-axis is
logarithmic, otherwise the fraud bars would be invisible. Oversampling (B, D) raises the fraud
bar, undersampling (C) lowers the legit bar, E does both, and the baseline (A) and class
weighting (F) leave the rows unchanged (F changes the weights instead). These are counts of
training rows only; validation and test rows are never resampled.

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(3.6 * len(models), 4), squeeze=False)
for ax, model in zip(axes[0], models, strict=True):
    names = [t for t in TECHNIQUES if f"{model}__{t}" in cells]
    after = [cells[f"{model}__{t}"]["train_counts_after"] for t in names]
    before = cells[f"{model}__{names[0]}"]["train_counts_before"]
    x = np.arange(len(names))
    ax.bar(x - 0.2, [a["legit"] for a in after], 0.4, label="legit after")
    ax.bar(x + 0.2, [a["fraud"] for a in after], 0.4, label="fraud after")
    ax.axhline(before["legit"], color="C0", linestyle="--", label="legit before")
    ax.axhline(before["fraud"], color="C1", linestyle="--", label="fraud before")
    ax.set_xticks(x, [LABELS[t] for t in names], rotation=45, ha="right")
    ax.set(yscale="log", title=model, ylabel="Training rows (log scale)")
axes[0][0].legend(fontsize=8)
fig.suptitle("Training class counts before and after each technique")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "training_counts.png", dpi=150)
plt.show()

## 3. Cross-validation results (training rows only)
One row per cell, best first: the mean and spread of PR-AUC over the repeated-CV folds, the
ratio(s) the search chose, and the F1 threshold chosen on out-of-fold scores. For a sampler the
ratio is "frauds = r x legit after resampling"; for class weighting it is the fraud weight w and
the ratio it matches (r = w x frauds / legit).

The cell at the top is the **headline cell**. It is chosen here, from training-only
cross-validation scores, so the test set plays no part in choosing it.

In [ ]:
def chosen_ratios(cell: dict) -> str:
    """The resampling ratio(s) or fraud weight the search picked for one cell, as short text."""
    counts = cell["train_counts_before"]
    base = counts["fraud"] / counts["legit"]
    parts = []
    for name, value in cell["best_params"].items():
        if name.endswith("sampling_strategy"):  # e.g. smote__sampling_strategy
            parts.append(f"{name.split('__')[0]} {value}")
        elif name == "model__scale_pos_weight":
            parts.append(f"fraud weight {value:.1f} (r = {value * base:.3g})")
        elif name == "model__class_weight" and value == "balanced":
            parts.append("balanced")
        elif name == "model__class_weight":
            weight = value["1"]  # JSON turned the class key 1 into the text "1"
            parts.append(f"fraud weight {weight:.1f} (r = {weight * base:.3g})")
    return ", ".join(parts) or "-"


cv_table = pd.DataFrame([{
    "model": cell["model"], "technique": LABELS[cell["technique"]],
    "CV PR-AUC mean": cell["cv_pr_auc_mean"], "CV PR-AUC std": cell["cv_pr_auc_std"],
    "chosen ratio(s)": chosen_ratios(cell), "F1 threshold": cell["threshold_f1"],
} for cell in cells.values()])
cv_table = cv_table.sort_values("CV PR-AUC mean", ascending=False, ignore_index=True)
headline = max(cells, key=lambda name: cells[name]["cv_pr_auc_mean"])
print(f"headline cell (highest CV PR-AUC): {headline}")
cv_table.round(4)

## 4. Test results
Every cell scored once on the test set it never saw, at the F1 threshold it chose on training
data. PR-AUC is the main metric; its 95% confidence interval comes from a stratified bootstrap
of the test rows, so overlapping intervals mean the test set cannot tell those cells apart.
ROC-AUC uses the scores only; precision, recall, F1 and F2 count the flagged payments at the
threshold.

**Accuracy is shown only to prove that it is useless here:** it is close to 99.9% for every cell,
because almost every payment is legit, and it would be about as high for a model that never
flags anything.

In [ ]:
rows_test = []
for model in models:
    for technique in TECHNIQUES:
        result = test_results.get(f"{model}__{technique}")
        if result is None:
            continue
        test, (low, high) = result["test"], result["pr_auc_ci"]
        rows_test.append({"model": model, "technique": LABELS[technique],
                          "test PR-AUC": test["pr_auc"], "95% CI": f"[{low:.3f}, {high:.3f}]",
                          "ROC-AUC": test["roc_auc"], "precision": test["precision"],
                          "recall": test["recall"], "F1": test["f1"], "F2": test["f2"],
                          "accuracy": test["accuracy"]})
test_table = pd.DataFrame(rows_test)
test_table.round(4)

## 5. Figures
**Precision-recall curves**, one panel per model and one line per technique. Read along the
x-axis: for each share of frauds caught (recall), the curve shows what share of the flagged
payments are really frauds (precision). Higher and further to the right is better; PR-AUC is the
area under this curve.

**Test PR-AUC with 95% bootstrap CI**: one group of bars per model, one bar per technique, the
whiskers show the confidence interval.

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(3.6 * len(models), 3.8), squeeze=False)
for ax, model in zip(axes[0], models, strict=True):
    for technique in TECHNIQUES:
        result = test_results.get(f"{model}__{technique}")
        if result:
            curve = result["pr_curve"]
            ax.plot(curve["recall"], curve["precision"], label=LABELS[technique])
    ax.set(title=model, xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1.02))
axes[0][0].legend(fontsize=8)
fig.suptitle("Precision-recall curves on the test set")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "pr_curves.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
width = 0.8 / len(TECHNIQUES)
for j, technique in enumerate(TECHNIQUES):
    for i, model in enumerate(models):
        result = test_results.get(f"{model}__{technique}")
        if result:
            value, (low, high) = result["test"]["pr_auc"], result["pr_auc_ci"]
            # (j - 2.5) * width spreads the 6 bars evenly around the model's tick at x = i
            ax.bar(i + (j - 2.5) * width, value, width, color=f"C{j}",
                   yerr=[[value - low], [high - value]], capsize=2,
                   label=LABELS[technique] if i == 0 else None)
ax.set_xticks(range(len(models)), models)
ax.set(ylabel="Test PR-AUC", ylim=(0, 1.05), title="Test PR-AUC with 95% bootstrap CI")
ax.legend(fontsize=8, ncol=3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "pr_auc_ci.png", dpi=150)
plt.show()

## 6. The headline cell in plain words
What the headline cell from section 3 would do with 100,000 payments at the fraud rate of
section 1, using its test recall (share of frauds caught) and false-positive rate (share of legit
payments flagged). The counts are rounded, so they are a picture of the rates, not exact
predictions.

In [ ]:
payments = 100_000
test = test_results[headline]["test"]
expected_frauds = payments * fraud_rate
caught = expected_frauds * test["recall"]
false_alarms = (payments - expected_frauds) * test["fpr"]
print(f"headline cell: {headline}, at its F1 threshold {test_results[headline]['threshold_f1']:.3f}")
print(f"out of {payments:,} payments, about {expected_frauds:.0f} are frauds")
print(f"  caught:  {caught:.0f}")
print(f"  missed:  {expected_frauds - caught:.0f}")
print(f"  legit payments wrongly flagged: {false_alarms:.0f}")

## 7. Significance tests
For each model, each technique is compared with the baseline on the repeated-CV fold scores (every
cell was scored on exactly the same folds, so they can be paired fold by fold).

- **Wilcoxon signed-rank test**: takes the fold-by-fold differences (technique minus baseline)
  and asks whether they lean to one side more than chance would explain. `median_difference`
  above 0 means the technique scored higher than the baseline on a typical fold.
- **Holm correction**: each model has 5 comparisons, and the more tests you run, the more
  likely one looks significant by luck. Holm raises the p-values to allow for that (the smallest
  is multiplied by 5, the next by 4, and so on). `significant` uses the Holm p-value and 0.05.
- **Friedman test**: ranks the 6 techniques within every fold and asks whether any of them ranks
  consistently higher or lower, i.e. whether the techniques differ at all for that model.

Only the models that have cells are shown, even if `stats.json` is older and lists others.

The p-values are **approximate**: the folds share training rows, so the fold scores are not
independent, while these tests assume they are. Treat them as a guide, not as proof.

In [ ]:
for model in models:  # only models with cells; stats.json may be older than the cells
    if model not in stats["models"]:
        print(f"{model}: no significance tests yet (run python -m fraudstudy.evaluation --stats)")
        continue
    result = stats["models"][model]
    table = pd.DataFrame(result["wilcoxon_vs_baseline"])
    table["technique"] = table["technique"].map(LABELS)
    table = table[["technique", "median_difference", "p_value", "p_value_holm", "significant"]]
    friedman_p = result["friedman"]["p_value"] if "friedman" in result else float("nan")
    print(f"{model}: Friedman p = {friedman_p:.3g}")
    display(table.round(4))

## 8. The README results table
The same numbers as Markdown text, to paste between the `RESULTS:START` and `RESULTS:END` marks
of `README.md`. CV PR-AUC is the training-only mean; the rest are test results at the cell's F1
threshold.

In [ ]:
lines = ["| Model | Technique | CV PR-AUC | Test PR-AUC (95% CI) | Precision | Recall | F1 threshold |",
         "|---|---|---|---|---|---|---|"]
for model in models:
    for technique in TECHNIQUES:
        result = test_results.get(f"{model}__{technique}")
        if result is None:
            continue
        test, (low, high) = result["test"], result["pr_auc_ci"]
        lines.append(f"| {model} | {LABELS[technique]} | {result['cv_pr_auc_mean']:.4f} "
                     f"| {test['pr_auc']:.4f} ({low:.3f}-{high:.3f}) | {test['precision']:.3f} "
                     f"| {test['recall']:.3f} | {result['threshold_f1']:.3f} |")
print("\n".join(lines))

## 9. Conclusions (written by the team)
To be written by the team after reading the results.